# 16. Black-Scholes 公式与 Greeks

## 学习目标

通过本次学习，你将能够：

1. **理解 BS 公式的假设与局限性**
2. **手写实现 BS 定价公式**（不依赖任何现成库）
3. **解释每个 Greek 的金融意义**
4. **绘制并解读 Greeks 随标的价格变化的曲线**

## 知识地图

```
BS 公式
├── 假设条件
│   ├── 标的资产价格服从几何布朗运动
│   ├── 无风险利率恒定
│   ├── 无摩擦市场（无交易成本、税收）
│   ├── 标的资产不支付股息
│   └── 欧式期权（只能在到期日行权）
├── 公式形式
│   ├── Call: C = S·N(d1) - K·e^(-rT)·N(d2)
│   └── Put: P = K·e^(-rT)·N(-d2) - S·N(-d1)
├── Greeks（风险敏感度）
│   ├── Delta (Δ): 价格敏感度
│   ├── Gamma (Γ): Delta 的变化率
│   ├── Theta (Θ): 时间衰减
│   ├── Vega (ν): 波动率敏感度
│   └── Rho (ρ): 利率敏感度
└── Put-Call Parity
    └── C + K·e^(-rT) = P + S
```

## 环境依赖

```bash
pip install numpy scipy matplotlib
```

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

---
## 1. 理论基础：Black-Scholes 公式

### 1.1 BS 公式的直觉理解

BS 公式的本质是：**期权价格 = 期望收益的现值**

在风险中性世界下，欧式看涨期权的价格为：

$$C = S \cdot N(d_1) - K \cdot e^{-rT} \cdot N(d_2)$$

其中：
- $S$：标的资产当前价格
- $K$：行权价
- $r$：无风险利率
- $T$：到期时间（年）
- $\sigma$：标的资产波动率
- $N(\cdot)$：标准正态分布的累积分布函数

$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$$

$$d_2 = d_1 - \sigma\sqrt{T}$$

### 1.2 公式的金融含义

| 项 | 含义 |
|---|---|
| $S \cdot N(d_1)$ | 持有标的资产的期望价值（Delta 对冲后） |
| $K \cdot e^{-rT} \cdot N(d_2)$ | 行权支出的期望现值 |
| $N(d_2)$ | 风险中性世界中期权被行权的概率 |

### 1.3 Put-Call Parity（看涨看跌平价关系）

$$C + K \cdot e^{-rT} = P + S$$

这个关系**不依赖任何模型假设**，只要无套利即可成立。

含义：**看涨期权 + 现金 = 看跌期权 + 股票**

---
## 2. 手写 BS 公式实现

### 2.1 核心实现

我们从零实现 BS 公式，只使用 `scipy.stats.norm` 计算正态分布 CDF。

In [ ]:
def bs_d1(S, K, r, sigma, T):
    """
    计算 BS 公式中的 d1
    
    Parameters
    ----------
    S : float - 标的资产当前价格
    K : float - 行权价
    r : float - 无风险利率（年化）
    sigma : float - 波动率（年化）
    T : float - 到期时间（年）
    
    Returns
    -------
    float : d1 值
    """
    return (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))


def bs_d2(S, K, r, sigma, T):
    """
    计算 BS 公式中的 d2
    """
    return bs_d1(S, K, r, sigma, T) - sigma * np.sqrt(T)


def bs_call(S, K, r, sigma, T):
    """
    计算欧式看涨期权价格
    
    BS 公式: C = S·N(d1) - K·e^(-rT)·N(d2)
    """
    d1 = bs_d1(S, K, r, sigma, T)
    d2 = bs_d2(S, K, r, sigma, T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def bs_put(S, K, r, sigma, T):
    """
    计算欧式看跌期权价格
    
    BS 公式: P = K·e^(-rT)·N(-d2) - S·N(-d1)
    """
    d1 = bs_d1(S, K, r, sigma, T)
    d2 = bs_d2(S, K, r, sigma, T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


# 测试我们的实现
S, K, r, sigma, T = 100, 100, 0.05, 0.2, 1  # 平值期权

call_price = bs_call(S, K, r, sigma, T)
put_price = bs_put(S, K, r, sigma, T)

print(f"参数设置：")
print(f"  标的价格 S = {S}")
print(f"  行权价   K = {K}")
print(f"  无风险利率 r = {r}")
print(f"  波动率   σ = {sigma}")
print(f"  到期时间 T = {T} 年")
print(f"\n计算结果：")
print(f"  看涨期权价格 C = {call_price:.4f}")
print(f"  看跌期权价格 P = {put_price:.4f}")

### 2.2 验证 Put-Call Parity

根据 Put-Call Parity：$C + K \cdot e^{-rT} = P + S$

两边应该相等。

In [ ]:
# 验证 Put-Call Parity
lhs = call_price + K * np.exp(-r * T)  # 左边: C + K·e^(-rT)
rhs = put_price + S                      # 右边: P + S

print(f"Put-Call Parity 验证：")
print(f"  左边 C + K·e^(-rT) = {lhs:.6f}")
print(f"  右边 P + S         = {rhs:.6f}")
print(f"  差异               = {abs(lhs - rhs):.10f}")
print(f"\n✓ Put-Call Parity 成立（差异为浮点误差）" if abs(lhs - rhs) < 1e-10 else "\n✗ 验证失败")

---
## 3. Greeks：期权风险敏感度

### 3.1 Greeks 的定义与金融意义

| Greek | 符号 | 定义 | 金融意义 |
|-------|------|------|----------|
| **Delta** | Δ | $\frac{\partial C}{\partial S}$ | 期权价格对标的资产价格的敏感度。Call: 0~1, Put: -1~0 |
| **Gamma** | Γ | $\frac{\partial^2 C}{\partial S^2}$ | Delta 对标的价格的敏感度。衡量 Delta 的稳定性 |
| **Theta** | Θ | $\frac{\partial C}{\partial t}$ | 期权价格随时间衰减的速度。通常为负值 |
| **Vega** | ν | $\frac{\partial C}{\partial \sigma}$ | 期权价格对波动率的敏感度。平值期权最大 |
| **Rho** | ρ | $\frac{\partial C}{\partial r}$ | 期权价格对无风险利率的敏感度 |

### 3.2 Greeks 的直觉理解

- **Delta**：如果你卖了一个 Call（Delta = 0.6），需要买入 0.6 份标的资产来对冲
- **Gamma**：Gamma 大意味着 Delta 变化快，需要更频繁地调整对冲
- **Theta**：期权是有期限的，时间流逝会减少期权价值（时间价值衰减）
- **Vega**：市场恐慌时波动率上升，期权（尤其是平值）会涨价
- **Rho**：利率上升时，Call 更值钱（行权支出的现值下降）

### 3.3 手写 Greeks 实现

我们通过**解析公式**计算 Greeks，这些公式可以通过对 BS 公式求导得到。

In [ ]:
def bs_greeks(S, K, r, sigma, T, option_type='call'):
    """
    计算 BS 期权的所有 Greeks
    
    Parameters
    ----------
    S : float - 标的价格
    K : float - 行权价
    r : float - 无风险利率
    sigma : float - 波动率
    T : float - 到期时间（年）
    option_type : str - 'call' 或 'put'
    
    Returns
    -------
    dict : 包含所有 Greeks 的字典
    """
    d1 = bs_d1(S, K, r, sigma, T)
    d2 = bs_d2(S, K, r, sigma, T)
    
    # 标准正态分布的 PDF
    nd1 = norm.pdf(d1)
    
    # 计算 Call 的 Greeks
    if option_type == 'call':
        delta = norm.cdf(d1)
        theta = (-S * nd1 * sigma / (2 * np.sqrt(T)) 
                 - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
        rho = K * T * np.exp(-r * T) * norm.cdf(d2) / 100
    # 计算 Put 的 Greeks
    elif option_type == 'put':
        delta = norm.cdf(d1) - 1
        theta = (-S * nd1 * sigma / (2 * np.sqrt(T)) 
                 + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100
    else:
        raise ValueError("option_type must be 'call' or 'put'")
    
    # Call 和 Put 的 Gamma 和 Vega 相同
    gamma = nd1 / (S * sigma * np.sqrt(T))
    vega = S * nd1 * np.sqrt(T) / 100  # 除以100表示波动率变化1%的影响
    
    return {
        'Delta': delta,
        'Gamma': gamma,
        'Theta': theta,
        'Vega': vega,
        'Rho': rho
    }


# 计算并展示 Greeks
print(f"平值期权的 Greeks（S={S}, K={K}, T={T}年, σ={sigma}, r={r}）：\n")

call_greeks = bs_greeks(S, K, r, sigma, T, 'call')
put_greeks = bs_greeks(S, K, r, sigma, T, 'put')

print("Call Greeks:")
for greek, value in call_greeks.items():
    print(f"  {greek:>8}: {value:>10.6f}")

print("\nPut Greeks:")
for greek, value in put_greeks.items():
    print(f"  {greek:>8}: {value:>10.6f}")

### 3.4 Greeks 的金融意义解读

让我们逐个理解每个 Greek：

#### Delta (Δ)
- **Call Delta = 0.6** 意味着：标的资产价格上涨 1 元，看涨期权价格大约上涨 0.6 元
- 也可以理解为：**风险中性世界中期权到期为实值的概率约为 60%**
- **Delta 对冲**：卖 Call 需要买入 Delta 份标的资产来对冲

#### Gamma (Γ)
- Gamma 是 Delta 的变化率，衡量 **Delta 的稳定性**
- Gamma 大 → Delta 变化快 → 需要更频繁地调整对冲
- **平值期权 Gamma 最大**，深度实值/虚值 Gamma 接近 0

#### Theta (Θ)
- Theta 通常为负值，表示**时间流逝会减少期权价值**
- **平值期权 Theta 最大**（时间价值衰减最快）
- 买期权 = 做多 Gamma，做空 Theta（Gamma 和 Theta 是对价关系）

#### Vega (ν)
- Vega 衡量**波动率变化对期权价格的影响**
- Vega 大 → 波动率上升 1%，期权价格上升 Vega 个点
- **平值期权 Vega 最大**

#### Rho (ρ)
- Rho 衡量**利率变化对期权价格的影响**
- 利率上升 → Call 更值钱（行权支出现值下降）
- 通常 Rho 影响较小，因为利率变化缓慢

---
## 4. 可视化：Greeks 随标的价格变化

### 4.1 绘制单个 Greek 的曲线

我们先绘制 Delta 随标的价格变化的曲线，来理解 Greeks 的形状。

In [ ]:
# 参数设置
K = 100       # 行权价
r = 0.05      # 无风险利率
sigma = 0.2   # 波动率
T = 0.5       # 到期时间 6 个月

# 标的价格范围：从 70 到 130
S_range = np.linspace(70, 130, 200)

# 计算 Call 和 Put 的 Delta
call_deltas = [bs_greeks(S, K, r, sigma, T, 'call')['Delta'] for S in S_range]
put_deltas = [bs_greeks(S, K, r, sigma, T, 'put')['Delta'] for S in S_range]

# 绘图
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(S_range, call_deltas, 'b-', linewidth=2, label='Call Delta')
ax.plot(S_range, put_deltas, 'r--', linewidth=2, label='Put Delta')
ax.axvline(x=K, color='gray', linestyle=':', alpha=0.5, label=f'行权价 K={K}')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=0.5, color='blue', linewidth=0.5, alpha=0.3)
ax.axhline(y=-0.5, color='red', linewidth=0.5, alpha=0.3)

ax.set_xlabel('标的价格 S', fontsize=12)
ax.set_ylabel('Delta', fontsize=12)
ax.set_title('Delta 随标的价格变化', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-1.1, 1.1)

plt.tight_layout()
plt.show()

print("\n解读：")
print("  - Call Delta 从 0（深度虚值）增加到 1（深度实值）")
print("  - Put Delta 从 0（深度虚值）减少到 -1（深度实值）")
print("  - 平值期权（S=K）的 Call Delta ≈ 0.5，Put Delta ≈ -0.5")
print("  - Delta 曲线在平值处最陡，说明平值期权对标的价格变化最敏感")

### 4.2 绘制所有 Greeks 的综合图

现在我们绘制 Call 期权的所有 Greeks 随标的价格变化的曲线。

In [ ]:
# 计算所有 Greeks
call_greeks_list = [bs_greeks(S, K, r, sigma, T, 'call') for S in S_range]

deltas = [g['Delta'] for g in call_greeks_list]
gammas = [g['Gamma'] for g in call_greeks_list]
thetas = [g['Theta'] for g in call_greeks_list]
vegas = [g['Vega'] for g in call_greeks_list]
rhos = [g['Rho'] for g in call_greeks_list]

# 创建 2x3 子图
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Delta
axes[0, 0].plot(S_range, deltas, 'b-', linewidth=2)
axes[0, 0].axvline(x=K, color='gray', linestyle=':', alpha=0.5)
axes[0, 0].axhline(y=0, color='black', linewidth=0.5)
axes[0, 0].set_title('Delta (Δ)', fontsize=12)
axes[0, 0].set_xlabel('标的价格 S')
axes[0, 0].set_ylabel('Delta')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim(-0.1, 1.1)

# Gamma
axes[0, 1].plot(S_range, gammas, 'r-', linewidth=2)
axes[0, 1].axvline(x=K, color='gray', linestyle=':', alpha=0.5)
axes[0, 1].set_title('Gamma (Γ)', fontsize=12)
axes[0, 1].set_xlabel('标的价格 S')
axes[0, 1].set_ylabel('Gamma')
axes[0, 1].grid(True, alpha=0.3)

# Theta
axes[0, 2].plot(S_range, thetas, 'g-', linewidth=2)
axes[0, 2].axvline(x=K, color='gray', linestyle=':', alpha=0.5)
axes[0, 2].axhline(y=0, color='black', linewidth=0.5)
axes[0, 2].set_title('Theta (Θ)', fontsize=12)
axes[0, 2].set_xlabel('标的价格 S')
axes[0, 2].set_ylabel('Theta (每日)')
axes[0, 2].grid(True, alpha=0.3)

# Vega
axes[1, 0].plot(S_range, vegas, 'm-', linewidth=2)
axes[1, 0].axvline(x=K, color='gray', linestyle=':', alpha=0.5)
axes[1, 0].set_title('Vega (ν)', fontsize=12)
axes[1, 0].set_xlabel('标的价格 S')
axes[1, 0].set_ylabel('Vega (σ+1%)')
axes[1, 0].grid(True, alpha=0.3)

# Rho
axes[1, 1].plot(S_range, rhos, 'c-', linewidth=2)
axes[1, 1].axvline(x=K, color='gray', linestyle=':', alpha=0.5)
axes[1, 1].axhline(y=0, color='black', linewidth=0.5)
axes[1, 1].set_title('Rho (ρ)', fontsize=12)
axes[1, 1].set_xlabel('标的价格 S')
axes[1, 1].set_ylabel('Rho (r+1%)')
axes[1, 1].grid(True, alpha=0.3)

# 隐藏最后一个子图
axes[1, 2].axis('off')

# 添加整体标题
fig.suptitle(f'Call 期权 Greeks 随标的价格变化\n(K={K}, σ={sigma}, r={r}, T={T}年)', 
             fontsize=14, y=1.02)

plt.tight_layout()
plt.show()

### 4.3 Greeks 形状的解读

| Greek | 形状特征 | 原因 |
|-------|----------|------|
| **Delta** | S 形曲线，从 0 到 1 | 深度虚值几乎不可能行权（Δ≈0），深度实值几乎肯定行权（Δ≈1） |
| **Gamma** | 钟形曲线，平值最高 | 平值期权的 Delta 对标的价格变化最敏感 |
| **Theta** | 倒钟形，平值最负 | 平值期权的时间价值最大，时间衰减也最快 |
| **Vega** | 钟形曲线，平值最高 | 平值期权对波动率变化最敏感 |
| **Rho** | 单调递增 | 标的价格越高，Call 越有可能行权，利率影响越大 |

**重要规律**：
- **Gamma、Theta、Vega 都在平值处最大**
- **Gamma 和 Theta 是对价关系**：Gamma 大意味着 Theta 也大（负值）
  - 买期权 = 做多 Gamma + 做空 Theta
  - 卖期权 = 做空 Gamma + 做多 Theta

### 4.4 不同到期时间的 Greeks 比较

时间对 Greeks 有什么影响？

In [ ]:
# 不同到期时间
T_list = [0.1, 0.25, 0.5, 1.0]  # 1个月、3个月、6个月、1年
colors = ['red', 'orange', 'green', 'blue']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Delta 比较
for T, color in zip(T_list, colors):
    deltas = [bs_greeks(S, K, r, sigma, T, 'call')['Delta'] for S in S_range]
    axes[0].plot(S_range, deltas, color=color, linewidth=2, label=f'T={T}年')

axes[0].axvline(x=K, color='gray', linestyle=':', alpha=0.5)
axes[0].set_title('Delta 随到期时间变化', fontsize=12)
axes[0].set_xlabel('标的价格 S')
axes[0].set_ylabel('Delta')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gamma 比较
for T, color in zip(T_list, colors):
    gammas = [bs_greeks(S, K, r, sigma, T, 'call')['Gamma'] for S in S_range]
    axes[1].plot(S_range, gammas, color=color, linewidth=2, label=f'T={T}年')

axes[1].axvline(x=K, color='gray', linestyle=':', alpha=0.5)
axes[1].set_title('Gamma 随到期时间变化', fontsize=12)
axes[1].set_xlabel('标的价格 S')
axes[1].set_ylabel('Gamma')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Theta 比较
for T, color in zip(T_list, colors):
    thetas = [bs_greeks(S, K, r, sigma, T, 'call')['Theta'] for S in S_range]
    axes[2].plot(S_range, thetas, color=color, linewidth=2, label=f'T={T}年')

axes[2].axvline(x=K, color='gray', linestyle=':', alpha=0.5)
axes[2].set_title('Theta 随到期时间变化', fontsize=12)
axes[2].set_xlabel('标的价格 S')
axes[2].set_ylabel('Theta (每日)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n解读：")
print("  - 到期时间越短，Delta 曲线越陡（在平值附近变化更快）")
print("  - 到期时间越短，Gamma 在平值处越大（Delta 更不稳定）")
print("  - 到期时间越短，Theta 在平值处越负（时间衰减越快）")
print("  - 这就是为什么「末日轮」期权波动特别大（Gamma 大）")

---
## 5. 实战应用：期权交易策略的 Greeks 分析

### 5.1 常见策略的 Greeks 特征

| 策略 | Delta | Gamma | Theta | Vega | 适用场景 |
|------|-------|-------|-------|------|----------|
| **买入 Call** | + | + | - | + | 看涨 |
| **买入 Put** | - | + | - | + | 看跌 |
| **卖出 Call** | - | - | + | - | 看不涨 |
| **卖出 Put** | + | - | + | - | 看不跌 |
| **买入跨式** | 0 | + | - | + | 预期大幅波动 |
| **卖出跨式** | 0 | - | + | - | 预期横盘整理 |

### 5.2 例子：分析一个跨式策略的 Greeks

In [ ]:
def straddle_greeks(S, K, r, sigma, T):
    """
    计算买入跨式策略（Long Straddle）的 Greeks
    
    策略：同时买入相同行权价的 Call 和 Put
    """
    call_greeks = bs_greeks(S, K, r, sigma, T, 'call')
    put_greeks = bs_greeks(S, K, r, sigma, T, 'put')
    
    return {
        'Delta': call_greeks['Delta'] + put_greeks['Delta'],
        'Gamma': call_greeks['Gamma'] + put_greeks['Gamma'],
        'Theta': call_greeks['Theta'] + put_greeks['Theta'],
        'Vega': call_greeks['Vega'] + put_greeks['Vega'],
        'Rho': call_greeks['Rho'] + put_greeks['Rho']
    }


# 计算跨式策略的 Greeks
K_straddle = 100
straddle_gs = straddle_greeks(S_range, K_straddle, r, sigma, T)

# 创建 2x2 子图
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Delta
axes[0, 0].plot(S_range, [straddle_greeks(S, K_straddle, r, sigma, T)['Delta'] for S in S_range], 
                'b-', linewidth=2)
axes[0, 0].axvline(x=K_straddle, color='gray', linestyle=':', alpha=0.5)
axes[0, 0].axhline(y=0, color='black', linewidth=0.5)
axes[0, 0].set_title('跨式策略 Delta', fontsize=12)
axes[0, 0].set_xlabel('标的价格 S')
axes[0, 0].grid(True, alpha=0.3)

# Gamma
axes[0, 1].plot(S_range, [straddle_greeks(S, K_straddle, r, sigma, T)['Gamma'] for S in S_range], 
                'r-', linewidth=2)
axes[0, 1].axvline(x=K_straddle, color='gray', linestyle=':', alpha=0.5)
axes[0, 1].set_title('跨式策略 Gamma', fontsize=12)
axes[0, 1].set_xlabel('标的价格 S')
axes[0, 1].grid(True, alpha=0.3)

# Theta
axes[1, 0].plot(S_range, [straddle_greeks(S, K_straddle, r, sigma, T)['Theta'] for S in S_range], 
                'g-', linewidth=2)
axes[1, 0].axvline(x=K_straddle, color='gray', linestyle=':', alpha=0.5)
axes[1, 0].axhline(y=0, color='black', linewidth=0.5)
axes[1, 0].set_title('跨式策略 Theta', fontsize=12)
axes[1, 0].set_xlabel('标的价格 S')
axes[1, 0].grid(True, alpha=0.3)

# Vega
axes[1, 1].plot(S_range, [straddle_greeks(S, K_straddle, r, sigma, T)['Vega'] for S in S_range], 
                'm-', linewidth=2)
axes[1, 1].axvline(x=K_straddle, color='gray', linestyle=':', alpha=0.5)
axes[1, 1].set_title('跨式策略 Vega', fontsize=12)
axes[1, 1].set_xlabel('标的价格 S')
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('买入跨式策略 Greeks 分析', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n跨式策略 Greeks 解读：")
print("  - Delta: 平值附近接近 0（方向中性），远离平值时 Delta 偏向 ±1")
print("  - Gamma: 平值处最大，说明在平值附近 Delta 变化最快")
print("  - Theta: 平值处最负，说明时间衰减在平值附近最严重")
print("  - Vega: 平值处最大，说明波动率上升对跨式策略最有利")
print("\n结论：跨式策略是做多波动率的策略，适合预期大幅波动但方向不确定时使用。")

---
## 6. 进阶：数值方法验证 Greeks

除了解析公式，我们还可以用**有限差分法**数值计算 Greeks，来验证我们的实现。

In [ ]:
def numerical_greeks(S, K, r, sigma, T, option_type='call', dS=0.01, dsigma=0.0001, dr=0.0001, dT=1/365):
    """
    用有限差分法数值计算 Greeks
    """
    price_func = bs_call if option_type == 'call' else bs_put
    
    # Delta: (C(S+dS) - C(S-dS)) / (2*dS)
    delta = (price_func(S+dS, K, r, sigma, T) - price_func(S-dS, K, r, sigma, T)) / (2 * dS)
    
    # Gamma: (C(S+dS) - 2*C(S) + C(S-dS)) / (dS^2)
    gamma = (price_func(S+dS, K, r, sigma, T) - 2*price_func(S, K, r, sigma, T) + price_func(S-dS, K, r, sigma, T)) / (dS**2)
    
    # Theta: (C(T-dT) - C(T)) / dT
    theta = (price_func(S, K, r, sigma, T-dT) - price_func(S, K, r, sigma, T)) / dT / 365
    
    # Vega: (C(sigma+dsigma) - C(sigma-dsigma)) / (2*dsigma)
    vega = (price_func(S, K, r, sigma+dsigma, T) - price_func(S, K, r, sigma-dsigma, T)) / (2 * dsigma) / 100
    
    # Rho: (C(r+dr) - C(r-dr)) / (2*dr)
    rho = (price_func(S, K, r+dr, sigma, T) - price_func(S, K, r-dr, sigma, T)) / (2 * dr) / 100
    
    return {
        'Delta': delta,
        'Gamma': gamma,
        'Theta': theta,
        'Vega': vega,
        'Rho': rho
    }


# 比较解析解和数值解
S_test = 105
analytical = bs_greeks(S_test, K, r, sigma, T, 'call')
numerical = numerical_greeks(S_test, K, r, sigma, T, 'call')

print(f"解析解 vs 数值解比较 (S={S_test}, K={K})：\n")
print(f"{'Greek':>8} {'解析解':>12} {'数值解':>12} {'差异':>12}")
print("-" * 50)
for greek in ['Delta', 'Gamma', 'Theta', 'Vega', 'Rho']:
    diff = abs(analytical[greek] - numerical[greek])
    print(f"{greek:>8} {analytical[greek]:>12.6f} {numerical[greek]:>12.6f} {diff:>12.8f}")

print("\n✓ 解析解和数值解高度一致，验证了我们的实现正确性！")

---
## 7. 小结

### 核心收获

1. **BS 公式**是期权定价的基石，虽然有严格的假设，但提供了重要的基准

2. **Greeks** 是期权交易的「仪表盘」：
   - Delta 告诉你方向风险
   - Gamma 告诉你 Delta 的稳定性
   - Theta 告诉你时间衰减的速度
   - Vega 告诉你波动率风险
   - Rho 告诉你利率风险

3. **平值期权**的 Gamma、Theta、Vega 都最大，这是期权交易的重要规律

4. **Gamma 和 Theta 是对价关系**：做多 Gamma 就要付出 Theta 的代价

### 关键公式

- Call: $C = S \cdot N(d_1) - K \cdot e^{-rT} \cdot N(d_2)$
- Put: $P = K \cdot e^{-rT} \cdot N(-d_2) - S \cdot N(-d_1)$
- Put-Call Parity: $C + K \cdot e^{-rT} = P + S$

### 延伸阅读

- [Options, Futures, and Other Derivatives](https://www.amazon.com/Options-Futures-Other-Derivatives-10th/dp/013447208X) - John Hull
- [The Greeks](https://en.wikipedia.org/wiki/Greeks_(finance)) - Wikipedia

---
## 验收标准 Checklist

完成本次学习后，你应该能够：

- [x] **能不用库写出 BS 定价**：我们从零实现了 `bs_call()` 和 `bs_put()` 函数
- [x] **能解释每个 Greek 的金融意义**：
  - Delta = 价格敏感度 / 行权概率
  - Gamma = Delta 的变化率 / 对冲频率
  - Theta = 时间衰减 / 期权是有保质期的
  - Vega = 波动率敏感度 / 恐慌指数
  - Rho = 利率敏感度 / 通常影响较小
- [x] **理解 Greeks 曲线的形状**：
  - Delta 是 S 形
  - Gamma/Theta/Vega 是钟形，平值处最大
  - 到期时间越短，Gamma/Theta 越极端

### 自测题

1. 一个 Call 的 Delta = 0.7，如果标的资产上涨 2 元，期权价格大约变化多少？
2. 为什么平值期权的 Gamma 最大？
3. 买入跨式策略（Long Straddle）的 Gamma 和 Theta 分别是什么符号？这意味着什么？
4. 如果你卖了一个 Call，你应该如何用标的资产来 Delta 对冲？

---

**恭喜你完成了 BS 公式与 Greeks 的学习！** 🎉

下次我们将学习**蒙特卡洛期权定价**，了解如何用随机模拟来定价更复杂的期权。